# Rule-Based AES: Google Colab Runner

This notebook runs the current grammar + anchored LLM rubric workflow on Google Colab. It writes new output files and does not overwrite the original workbooks.

## 1. Choose A Runtime

In Colab, choose `Runtime > Change runtime type`, then pick a GPU runtime if available. The workflow can run on CPU, but `llama3:8b` with six anchors per essay is much slower.

In [ ]:
# If you opened this notebook outside the cloned repo, clone it first.
# If the repo is already present in /content/Rule-Based-AES, this cell leaves it alone.
import os
from pathlib import Path

repo_dir = Path('/content/Rule-Based-AES')
if not repo_dir.exists():
    !git clone https://github.com/EPHRAIMTEODORO/Rule-Based-AES.git /content/Rule-Based-AES
%cd /content/Rule-Based-AES
print('Repo:', Path.cwd())

## 2. Install Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jre-headless curl > /dev/null
!pip install -q -r "GColab Version/requirements_colab.txt"
!java -version

## 3. Install And Start Ollama

This installs Ollama inside the Colab runtime, starts the server in the background, and pulls `llama3:8b`.

In [ ]:
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request, json, os, signal

def ollama_ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_ready():
    ollama_log = open('/content/ollama_colab.log', 'w')
    ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(2)

print('Ollama ready:', ollama_ready())

In [ ]:
!ollama pull llama3:8b
!ollama list

## 4. Confirm Input Files

In [ ]:
from pathlib import Path

essays = Path('NewAes/ELAT_DATA/essays.xlsx')
base_features = Path('NewAes/ELAT_DATA/full_sample_aes_features.xlsx')
grammar_features = Path('NewAes/ELAT_DATA/full_sample_aes_features_with_grammar.xlsx')
llm_output = Path('NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_llm_improved_colab.xlsx')

for path in [essays, base_features, grammar_features]:
    print(path, 'exists=', path.exists())

## 5. Optional: Generate Grammar Features

Run this only if `full_sample_aes_features_with_grammar.xlsx` is missing or you want to regenerate it.

In [ ]:
!python "GColab Version/colab_add_grammar_features_to_full_sample.py" \
  --essays-file "NewAes/ELAT_DATA/essays.xlsx" \
  --features-file "NewAes/ELAT_DATA/full_sample_aes_features.xlsx" \
  --output-file "NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_colab.xlsx" \
  --language-tool-version 6.8

print('Grammar Colab output: NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_colab.xlsx')

## 6. Run Improved Anchored LLM Features

This uses resume mode and saves after every essay. If Colab disconnects, rerun setup and this same cell.

In [ ]:
!python "GColab Version/colab_llm_rubric_features.py" \
  --features-file "NewAes/ELAT_DATA/full_sample_aes_features_with_grammar.xlsx" \
  --output "NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_llm_improved_colab.xlsx" \
  --resume \
  --timeout 240 \
  --retries 2 \
  --retry-delay-seconds 10 \
  --save-every 1 \
  --delay-seconds 1

## 7. Check Progress Or Final Completion

In [ ]:
from openpyxl import load_workbook
from pathlib import Path

output = Path('NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_llm_improved_colab.xlsx')
print('exists:', output.exists())
if output.exists():
    wb = load_workbook(output, read_only=True, data_only=True)
    ws = wb.active
    headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
    overall_idx = headers.index('llm_overall_score')
    total = ws.max_row - 1
    completed = sum(1 for row in ws.iter_rows(min_row=2, values_only=True) if row[overall_idx] is not None)
    wb.close()
    print('total:', total)
    print('completed:', completed)
    print('left:', total - completed)

## 8. Download Output

In [ ]:
from google.colab import files
files.download('NewAes/ELAT_DATA/full_sample_aes_features_with_grammar_llm_improved_colab.xlsx')